# term-project

6주차 planning_agent 코드를 langgraph로 바꾸기


## 0. 초기화

In [1]:
import os
import json
import tempfile
from pathlib import Path
from copy import deepcopy

assert (Path.cwd() / "agent" / "core" / "graph.py").exists(), (
    "노트북을 프로젝트 app/ 루트에서 실행해야 합니다. 현재 경로: " + str(Path.cwd())
)
from dotenv import load_dotenv
from pathlib import Path
import os

load_dotenv(Path.cwd() / ".env", override=True)

from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command

from agent.core.graph import build_graph, create_turn_input
from agent.core.llm import create_client
from agent.plan.common import RECENT_TURNS
from agent.tools import DOCUMENTS_FILE, load_memories, reset_memory_store

client = create_client()
model = os.getenv("MODEL_NAME") or client.models.list().data[0].id
checkpointer = InMemorySaver()
app = build_graph(
    client=client,
    model=model,
    recent_turns=RECENT_TURNS,
    use_summary=True,
    debug=False,
    checkpointer=checkpointer,
)
reset_memory_store()

assert DOCUMENTS_FILE.is_file(), f"문서 파일이 없습니다: {DOCUMENTS_FILE}"
print("Model:", model)
print("Documents:", DOCUMENTS_FILE, "| loaded:", len(json.loads(DOCUMENTS_FILE.read_text(encoding="utf-8"))))
print("Memory file: ", os.getenv("MEMORY_FILE"))
print("Summary recent_turns:", RECENT_TURNS)

h:\p2026\term_agent\app\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading documents from: H:\p2026\term_agent\app\data\documents.json
Model: qwen3.5-9b
Documents: H:\p2026\term_agent\app\data\documents.json | loaded: 6
Memory file:  data/memory_store.json
Summary recent_turns: 2


### 공통 출력·실행 헬퍼


In [2]:
def thread_config(thread_id: str) -> dict:
    return {"configurable": {"thread_id": thread_id}, "recursion_limit": 100}


def review_payload(result: dict) -> dict | None:
    interrupts = result.get("__interrupt__", ())
    return interrupts[0].value if interrupts else None


def print_review(result: dict, *, title: str = "계획 검토") -> None:
    review = review_payload(result)
    if review is None:
        print(f"[{title}] interrupt 없음")
        return
    print(f"\n[{title}]")
    print("목표:", review["goal"])
    print("질문:", review["question"])
    for idx, step in enumerate(review["plan"], start=1):
        print(f"{idx}. {step['instruction']}")
        print("   depends_on:", step["depends_on"])


def print_trace(result: dict) -> None:
    task = result.get("current_task") or {}
    for event in task.get("trace", []):
        print(json.dumps(event, ensure_ascii=False, default=str))


def print_outcome(result: dict, *, show_trace: bool = True) -> None:
    task = result.get("current_task") or {}
    print(f"\n[Turn status] {result.get('turn_status')}")
    print("[Final Answer]\n", result.get("final_response"))
    print("\n[Task state]")
    print("Replans:", task.get("replan_count"))
    print("Counted JSON LLM calls (not every LLM request):", task.get("llm_call_count"))
    print("Aborted reason:", task.get("aborted_reason"))
    print("Progress:", result.get("progress"))
    print("Turns:", len(result.get("turns", [])))
    print("\n[PlanStep statuses]")
    for step in task.get("plan", []):
        print(f"Step {step['id']}: {step['status']} | depends_on={step['depends_on']}")
    if show_trace:
        print("\n[Trace]")
        print_trace(result)


def start_turn(question: str, config: dict, *, continue_thread: bool = False) -> dict:
    previous_state = None
    if continue_thread:
        previous_state = app.get_state(config).values
        if not previous_state:
            raise ValueError("이어갈 Checkpoint가 없습니다. 선행 셀을 실행하세요.")
    return app.invoke(create_turn_input(question, previous_state), config=config)


def resume_approval(result: dict, config: dict) -> dict:
    if review_payload(result) is None:
        print("Plan Review에 도달하지 않았습니다. 현재 종료 상태를 확인하세요.")
        return result
    return app.invoke(Command(resume={"approved": True}), config=config)


def run_approved_turn(question: str, config: dict, *, continue_thread: bool = False) -> dict:
    result = start_turn(question, config, continue_thread=continue_thread)
    count = 0
    while review_payload(result) is not None:
        print_review(result)
        print("⇒ Command(resume={'approved': True})")
        result = resume_approval(result, config)
        count += 1
        if count > 5:
            raise RuntimeError("Plan Review가 예상보다 많이 반복됩니다.")
    return result


def checkpoint_details(config: dict, label: str) -> dict:
    snapshot = app.get_state(config)
    values = snapshot.values
    print(f"\n[{label}]")
    print("thread_id:", config["configurable"]["thread_id"])
    print("checkpoint_id:", snapshot.config.get("configurable", {}).get("checkpoint_id"))
    print("next nodes:", snapshot.next)
    print("turn_status:", values.get("turn_status"))
    print("completed turns:", len(values.get("turns", [])))
    print("current plan steps:", len((values.get("current_task") or {}).get("plan", [])))
    return deepcopy(values)

## 1. 도구 능력 밖의 질문

**질문:** 서울의 오늘 날씨. 이 에이전트는 실시간 날씨 Tool이 없으므로, 내부 문서 검색만으로 실제 날씨를 확인한 것처럼 단정하면 안 된다. 계획 생성 및 최종 응답, Trace를 살펴본다.

In [3]:
weather_config = thread_config("week7-outside-tools")
weather_question = "서울의 오늘 날씨를 알려줘요"
weather_result = run_approved_turn(weather_question, weather_config)
print_outcome(weather_result)


[계획 검토]
목표: 서울의 오늘 날씨를 알려줘요
질문: 이 계획을 승인하거나 수정할 내용을 입력하세요.
1. Determine that the user's request for today's weather in Seoul cannot be fulfilled because the available Executor capabilities (searching internal company documents, reading/writing user memory, reasoning over evidence) do not include web search, weather data access, or any external tools.
   depends_on: []
⇒ Command(resume={'approved': True})


Loading weights: 100%|██████████| 310/310 [00:00<00:00, 1146.99it/s]
This model was loaded in torch.bfloat16 on CPU. Depending on your CPU, inference may be significantly slower than with torch.float32. Consider comparing inference speeds by loading with `model_kwargs={"dtype": torch.float32}`.



[Turn status] completed
[Final Answer]
 죄송합니다만, 현재 내장된 실행 기능 (내부 문서 검색, 사용자 메모 읽기/쓰기, 증거에 대한 추론) 에는 웹 검색이나 날씨 데이터 접근 기능이 포함되어 있지 않아 서울의 오늘 날씨를 확인해 드릴 수 없습니다.

[Task state]
Replans: 0
Counted JSON LLM calls (not every LLM request): 2
Aborted reason: None
Progress: {'decision': 'finish', 'reason': 'The current step has successfully determined that the request cannot be fulfilled due to capability limitations, and there are no remaining steps in the plan.'}
Turns: 1

[PlanStep statuses]
Step 1: completed | depends_on=[]

[Trace]
{"event": "turn_started", "goal": "서울의 오늘 날씨를 알려줘요"}
{"event": "plan_created", "plan": [{"id": 1, "instruction": "Determine that the user's request for today's weather in Seoul cannot be fulfilled because the available Executor capabilities (searching internal company documents, reading/writing user memory, reasoning over evidence) do not include web search, weather data access, or any external tools.", "depends_on": [], "status": "pending"}]}
{"event": "plan_app

In [4]:
memories = load_memories()
print("Stored test memories:", len(memories))
for memory in memories:
    print("[Memory]", memory.get("content"))

Stored test memories: 1
[Memory] The user requested today's weather in Seoul.


## 2. 정상 과제 — 문서 검색 및 계획 승인 테스트

**기존 6주차 정상 과제**를 새로운 `StateGraph`로 실행한다. 동시에 과제에서 요구한 시나리오 A(생성된 계획 그대로 승인)도 검증한다. 계획 검토에서 `interrupt()`가 발생하고, `Command(resume={"approved": True})`로 동일 `thread_id`를 재개한다.

In [5]:
TASK_QUESTION = (
    "2026년 해외 출장의 숙박비와 식비 한도, 그리고 출발 전 승인 요건을 "
    "근거와 함께 정리해줘."
)

approval_config = thread_config("week7-travel-approval")
approval_paused = start_turn(TASK_QUESTION, approval_config)
assert review_payload(approval_paused) is not None, (
    "정상 과제의 Planner가 Plan Review에 도달하지 않았습니다. Trace를 점검하세요."
)
print_review(approval_paused, title="시나리오 A — 승인 전 Plan Review")
approval_plan_before = deepcopy(approval_paused["current_task"]["plan"])

# 승인 전 Checkpoint(중단 위치와 Plan 보존) 확보
approval_before_snapshot = app.get_state(approval_config)
approval_checkpoint_before = checkpoint_details(approval_config, "승인 전 Checkpoint")
assert approval_checkpoint_before["current_task"]["plan"] == approval_plan_before
assert approval_checkpoint_before["turn_status"] == "running"


[시나리오 A — 승인 전 Plan Review]
목표: 2026년 해외 출장의 숙박비와 식비 한도, 그리고 출발 전 승인 요건을 근거와 함께 정리해줘.
질문: 이 계획을 승인하거나 수정할 내용을 입력하세요.
1. 검색: 2026 년 해외 출장 관련 정책 문서에서 숙박비와 식비의 한도 기준을 찾아라.
   depends_on: []
2. 검색: 2026 년 해외 출장 관련 정책 문서에서 출발 전 승인 요건과 근거를 찾아라.
   depends_on: []
3. 추론: 검색된 숙박비, 식비 한도 및 출발 전 승인 요건 정보를 종합하여 근거와 함께 정리하라.
   depends_on: [1, 2]

[승인 전 Checkpoint]
thread_id: week7-travel-approval
checkpoint_id: 1f1d69af-855f-61ae-8001-4a2e09ba8263
next nodes: ('plan_review',)
turn_status: running
completed turns: 0
current plan steps: 3


In [6]:
# 사용자 승인을 코드로 명시적으로 전달. 같은 thread_id로 실행을 재개한다.
approval_result = app.invoke(
    Command(resume={"approved": True}),
    config=approval_config,
)
assert review_payload(approval_result) is None, "승인했는데 다시 중단되었습니다."
approval_event_names = [e["event"] for e in approval_result["current_task"]["trace"]]
assert "plan_approved" in approval_event_names, "승인 Trace가 기록되지 않았습니다."
print_outcome(approval_result)


[Turn status] completed
[Final Answer]
 2026 년 해외 출장 관련 정책 정보를 근거와 함께 정리하면 다음과 같습니다.

**1. 숙박비 한도**
*   **기준**: 국제 비즈니스 출장 시 숙박비 청구 한도는 **박당 220 달러**입니다.
*   **근거**: `travel_lodging_B.txt` (2026 년, active 상태)

**2. 식비 한도**
*   **기준**: 승인된 비즈니스 출장 시 식비 보전 한도는 **일당 60 달러**입니다.
*   **근거**: `meal_expense.txt` (2026 년, active 상태)

**3. 출발 전 승인 요건**
*   **요건**: 국제 비즈니스 출장 요청은 **출발 전에 부서장 (department head) 의 승인**을 받아야 합니다.
*   **근거**: `travel_request.txt` (2026 년, active 상태)

상기 내용은 2026 년 현재 적용 중인 내부 정책 문서에서 확인된 사항입니다.

[Task state]
Replans: 0
Counted JSON LLM calls (not every LLM request): 4
Aborted reason: None
Progress: {'decision': 'finish', 'reason': 'The current step has successfully synthesized the required information (accommodation limit, meal limit, and pre-departure approval requirements) with supporting evidence from the retrieved documents. No remaining steps are needed.'}
Turns: 1

[PlanStep statuses]
Step 1: completed | depends_on=[]
Step 2: completed | depends_on=[]
Step 3: 

## 3. Checkpoint 검증 — 중단 전후 State 비교

`InMemorySaver`와 `thread_id`로 계획 생성 후 중단 당시의 State가 보존되고, 계획 생성 후 중단된 Checkpoint와 승인 직후(Executor 실행 전) Checkpoint를 비교한다. 동일한 thread에서 두 Checkpoint가 연결되는지, 목표와 계획이 유지되고 승인 처리가 반영됐는지 확인한다.

In [7]:
from copy import deepcopy
from html import escape
from IPython.display import Markdown, display

# 중단 당시 저장된 Checkpoint를 ID로 다시 조회한다.
paused = app.get_state(approval_before_snapshot.config)
paused_id = paused.config["configurable"]["checkpoint_id"]

# 중단 Checkpoint의 바로 다음 상태: 승인 처리 완료, Executor 실행 전.
approved = next(
    (
        snapshot
        for snapshot in app.get_state_history(approval_config)
        if (snapshot.parent_config or {})
            .get("configurable", {})
            .get("checkpoint_id") == paused_id
        and snapshot.next == ("executor",)
    ),
    None,
)
assert approved is not None, "승인 직후 Checkpoint가 없습니다. 2번 승인 셀을 먼저 실행하세요."

before = paused.values
after = approved.values
before_task = before["current_task"]
after_task = after["current_task"]

assert before == approval_checkpoint_before, "저장된 중단 State가 다릅니다."
assert paused.config["configurable"]["thread_id"] == approved.config["configurable"]["thread_id"]
assert paused_id != approved.config["configurable"]["checkpoint_id"]
assert paused.next == ("plan_review",)
assert before["turn_status"] == after["turn_status"] == "running"
assert before_task["goal"] == after_task["goal"], "목표가 변경되었습니다."
assert before_task["plan"] == after_task["plan"], "승인 과정에서 계획이 변경되었습니다."
assert before["turns"] == after["turns"], "승인 직후에 이미 Turn이 추가되었습니다."
assert before.get("step_result") == after.get("step_result") is None
assert before.get("final_response") == after.get("final_response") is None
assert after_task["trace"][-1]["event"] == "plan_approved"


def plan_text(task):
    return "\n".join(
        f"{step['id']}. {step['instruction']}\n"
        f"상태: {step['status']} / 선행 단계: {step['depends_on']}"
        for step in task["plan"]
    )


def cell_text(value):
    if value is None:
        return "없음"
    return escape(str(value)).replace("|", "\\|").replace("\n", "<br>")


rows = [
    ("thread_id",
     paused.config["configurable"]["thread_id"],
     approved.config["configurable"]["thread_id"]),
    ("checkpoint_id",
     paused_id,
     approved.config["configurable"]["checkpoint_id"]),
    ("parent checkpoint_id",
     (paused.parent_config or {}).get("configurable", {}).get("checkpoint_id"),
     approved.parent_config["configurable"]["checkpoint_id"]),
    ("다음 노드", ", ".join(paused.next), ", ".join(approved.next)),
    ("turn_status", before["turn_status"], after["turn_status"]),
    ("목표", before_task["goal"], after_task["goal"]),
    ("계획", plan_text(before_task), plan_text(after_task)),
    ("현재 PlanStep",
     before_task["current_plan_step_id"],
     after_task["current_plan_step_id"]),
    ("마지막 이벤트",
     before_task["trace"][-1]["event"],
     after_task["trace"][-1]["event"]),
    ("완료된 Turn 수", len(before["turns"]), len(after["turns"])),
    ("최종 답변", before.get("final_response"), after.get("final_response")),
]

display(Markdown(
    "| 항목 | 중단 시점 | 승인 직후 — Executor 실행 전 |\n"
    "|---|---|---|\n"
    + "\n".join(
        f"| {name} | {cell_text(left)} | {cell_text(right)} |"
        for name, left, right in rows
    )
))

# 기존 6번 최종 점검용 변수. 위 비교에는 사용하지 않는다.
approval_checkpoint_after = deepcopy(approval_result)

| 항목 | 중단 시점 | 승인 직후 — Executor 실행 전 |
|---|---|---|
| thread_id | week7-travel-approval | week7-travel-approval |
| checkpoint_id | 1f1d69af-855f-61ae-8001-4a2e09ba8263 | 1f1d69af-8572-637e-8002-d1a71d6244fc |
| parent checkpoint_id | 1f1d69af-16be-6323-8000-6800b6a80308 | 1f1d69af-855f-61ae-8001-4a2e09ba8263 |
| 다음 노드 | plan_review | executor |
| turn_status | running | running |
| 목표 | 2026년 해외 출장의 숙박비와 식비 한도, 그리고 출발 전 승인 요건을 근거와 함께 정리해줘. | 2026년 해외 출장의 숙박비와 식비 한도, 그리고 출발 전 승인 요건을 근거와 함께 정리해줘. |
| 계획 | 1. 검색: 2026 년 해외 출장 관련 정책 문서에서 숙박비와 식비의 한도 기준을 찾아라.<br>상태: pending / 선행 단계: []<br>2. 검색: 2026 년 해외 출장 관련 정책 문서에서 출발 전 승인 요건과 근거를 찾아라.<br>상태: pending / 선행 단계: []<br>3. 추론: 검색된 숙박비, 식비 한도 및 출발 전 승인 요건 정보를 종합하여 근거와 함께 정리하라.<br>상태: pending / 선행 단계: [1, 2] | 1. 검색: 2026 년 해외 출장 관련 정책 문서에서 숙박비와 식비의 한도 기준을 찾아라.<br>상태: pending / 선행 단계: []<br>2. 검색: 2026 년 해외 출장 관련 정책 문서에서 출발 전 승인 요건과 근거를 찾아라.<br>상태: pending / 선행 단계: []<br>3. 추론: 검색된 숙박비, 식비 한도 및 출발 전 승인 요건 정보를 종합하여 근거와 함께 정리하라.<br>상태: pending / 선행 단계: [1, 2] |
| 현재 PlanStep | 1 | 1 |
| 마지막 이벤트 | plan_created | plan_approved |
| 완료된 Turn 수 | 0 | 0 |
| 최종 답변 | 없음 | 없음 |

## 4. 계획 **수정** 테스트 — 자연어 → LLM 재작성 → 재검토 → 승인

**시나리오 B:** 같은 출장 질문을 별도 `thread_id`에서 다시 실행한다. 최초 `interrupt` 후 자연어 피드백을 전달하면, Graph는 `plan_editor`를 실행해 새 계획을 생성하고 **다시 `plan_review`에서 중단**한다. 이를 확인한 뒤 승인하여 실제 수정 계획대로 수행한다.

> 수정 지시는 초기 계획의 세부 구성이 달라도 비교할 수 있도록 작업 구조를 명시한다. LLM 출력은 실행마다 달라질 수 있다.

In [8]:
edit_config = thread_config("week7-travel-edit")
edit_paused = start_turn(TASK_QUESTION, edit_config)
assert review_payload(edit_paused) is not None, "수정 테스트의 첫 interrupt가 없습니다."
print_review(edit_paused, title="시나리오 B — 수정 전 Plan")
edit_plan_before = deepcopy(edit_paused["current_task"]["plan"])
edit_before_checkpoint = checkpoint_details(edit_config, "수정 요청 전 Checkpoint")


[시나리오 B — 수정 전 Plan]
목표: 2026년 해외 출장의 숙박비와 식비 한도, 그리고 출발 전 승인 요건을 근거와 함께 정리해줘.
질문: 이 계획을 승인하거나 수정할 내용을 입력하세요.
1. 검색: 2026 년 해외 출장 관련 정책 문서에서 숙박비와 식비의 한도 기준을 찾아라.
   depends_on: []
2. 검색: 2026 년 해외 출장 관련 정책 문서에서 출발 전 승인 요건과 근거를 찾아라.
   depends_on: []
3. 추론: 검색된 숙박비, 식비 한도 및 출발 전 승인 요건 정보를 종합하여 근거와 함께 정리하라.
   depends_on: [1, 2]

[수정 요청 전 Checkpoint]
thread_id: week7-travel-edit
checkpoint_id: 1f1d69b2-b9b3-6695-8001-6717cbc816db
next nodes: ('plan_review',)
turn_status: running
completed turns: 0
current plan steps: 3


In [9]:
natural_feedback = (
    "현재 숙박비와 식비를 함께 검색하는 단계를 분리해줘. "
    "계획을 총 4단계로 수정해. "
    "1단계는 숙박비 한도 검색, "
    "2단계는 식비 한도 검색, "
    "3단계는 출발 전 승인 요건 검색, "
    "4단계는 1, 2, 3번의 결과와 근거 파일명을 종합하도록 해줘. "
    "마지막 단계의 depends_on은 [1, 2, 3]으로 설정해."
)

print("[사용자 자연어 수정 요청]", natural_feedback)

edit_repaused = app.invoke(
    Command(resume={"approved": False, "feedback": natural_feedback}),
    config=edit_config,
)
assert review_payload(edit_repaused) is not None, (
    "수정 후 다시 Plan Review에 중단하지 않았습니다. Plan Editor Trace를 확인하세요."
)
print_review(edit_repaused, title="LLM 수정 후 재검토 계획")
edit_plan_after = deepcopy(edit_repaused["current_task"]["plan"])
edit_during_checkpoint = checkpoint_details(edit_config, "수정 후 재검토 Checkpoint")

print("\n[계획 수정 비교]")
print("수정 전 Step 수:", len(edit_plan_before), "→ 수정 후:", len(edit_plan_after))
print("수정 전 instruction:", [s["instruction"] for s in edit_plan_before])
print("수정 후 instruction:", [s["instruction"] for s in edit_plan_after])
print("변경 여부:", edit_plan_before != edit_plan_after)
assert edit_during_checkpoint["current_task"]["plan_revision"] >= 1
assert any(e["event"] == "plan_edited" for e in edit_during_checkpoint["current_task"]["trace"])
assert any(e["event"] == "plan_revision_requested" for e in edit_during_checkpoint["current_task"]["trace"])

[사용자 자연어 수정 요청] 현재 숙박비와 식비를 함께 검색하는 단계를 분리해줘. 계획을 총 4단계로 수정해. 1단계는 숙박비 한도 검색, 2단계는 식비 한도 검색, 3단계는 출발 전 승인 요건 검색, 4단계는 1, 2, 3번의 결과와 근거 파일명을 종합하도록 해줘. 마지막 단계의 depends_on은 [1, 2, 3]으로 설정해.

[LLM 수정 후 재검토 계획]
목표: 2026년 해외 출장의 숙박비와 식비 한도, 그리고 출발 전 승인 요건을 근거와 함께 정리해줘.
질문: 이 계획을 승인하거나 수정할 내용을 입력하세요.
1. 검색: 2026 년 해외 출장 관련 정책 문서에서 숙박비 한도 기준을 찾아라.
   depends_on: []
2. 검색: 2026 년 해외 출장 관련 정책 문서에서 식비 한도 기준을 찾아라.
   depends_on: []
3. 검색: 2026 년 해외 출장 관련 정책 문서에서 출발 전 승인 요건과 근거를 찾아라.
   depends_on: []
4. 추론: 검색된 숙박비, 식비 한도 및 출발 전 승인 요건 정보를 종합하여 근거와 함께 정리하라.
   depends_on: [1, 2, 3]

[수정 후 재검토 Checkpoint]
thread_id: week7-travel-edit
checkpoint_id: 1f1d69b3-476a-64ec-8003-8a6f30eca8e7
next nodes: ('plan_review',)
turn_status: running
completed turns: 0
current plan steps: 4

[계획 수정 비교]
수정 전 Step 수: 3 → 수정 후: 4
수정 전 instruction: ['검색: 2026 년 해외 출장 관련 정책 문서에서 숙박비와 식비의 한도 기준을 찾아라.', '검색: 2026 년 해외 출장 관련 정책 문서에서 출발 전 승인 요건과 근거를 찾아라.', '추론: 검색된 숙박비, 식비 한도 및 출발 전 승인 요건 정보를 종합하여 근거와 함께 정리하라.']
수정 후 instruc

In [10]:
# 수정된 계획을 두 번째 검토에서 승인하고 실제 Executor 실행을 확인한다.
edit_result = app.invoke(
    Command(resume={"approved": True}),
    config=edit_config,
)
assert review_payload(edit_result) is None, "수정 계획 승인 후 Graph가 다시 중단했습니다."
edit_events = [event["event"] for event in edit_result["current_task"]["trace"]]
assert {"plan_revision_requested", "plan_edited", "plan_approved"}.issubset(edit_events)
print_outcome(edit_result)

print("\n[수정한 계획이 실행되었는가?]")
executed_ids = [
    e["plan_step_id"] for e in edit_result["current_task"]["trace"]
    if e["event"] == "plan_step_started"
]
print("실제 시작된 PlanStep IDs:", executed_ids)
print("수정 계획 Step IDs:", [s["id"] for s in edit_plan_after])
print("최종 Turn status:", edit_result["turn_status"])
print("수정 이후 자동 Replan 횟수:", edit_result["current_task"]["replan_count"])
# 자동 Replan이 발생하면 새로운 Step ID가 추가될 수 있으므로 ID 완전 일치는 강제하지 않는다.
assert executed_ids, "수정 승인 후 실행된 PlanStep이 없습니다."
assert executed_ids[0] == edit_plan_after[0]["id"]


[Turn status] completed
[Final Answer]
 2026 년 해외 출장 관련 정책 정보를 근거와 함께 정리하면 다음과 같습니다.

**1. 숙박비 한도**
*   **기준**:每晚 (하루) 최대 **220 달러**까지 보상 가능합니다.
*   **근거**: `travel_lodging_B.txt` (2026 년, active 상태)

**2. 식비 한도**
*   **기준**: 하루 최대 **60 달러**까지 보상됩니다.
*   **근거**: `meal_expense.txt` (2026 년, active 상태)

**3. 출발 전 승인 요건**
*   **요건**: 국제 비즈니스 출장 요청은 출발 전에 **부서장 (department head)**의 승인을 받아야 합니다.
*   **근거**: `travel_request.txt` (2026 년, active 상태)

모든 정보는 2026 년 현재 활성화된 내부 정책 문서에서 확인된 내용입니다.

[Task state]
Replans: 0
Counted JSON LLM calls (not every LLM request): 6
Aborted reason: None
Progress: {'decision': 'finish', 'reason': '추론 단계가 완료되었으며, 필요한 정보 (숙박비, 식비 한도, 승인 요건) 가 문서에서 확인되어 종합 정리되었습니다. 남은 단계가 없습니다.'}
Turns: 1

[PlanStep statuses]
Step 1: completed | depends_on=[]
Step 2: completed | depends_on=[]
Step 3: completed | depends_on=[]
Step 4: completed | depends_on=[1, 2, 3]

[Trace]
{"event": "turn_started", "goal": "2026년 해외 출장의 숙박비와 식비 한도, 그리고 출발 전 승인 요건을 근거와 함께 정리해줘."}
{"event": "pl

## 5. Summary 테스트

`create_turn_input(question, previous_state)`가 이전 `turns`와 `summary`를 보존하며, 기본 `RECENT_TURNS=2`에서는 완료 Turn이 3개가 되었을 때 Summary LLM 호출이 발생한다. 
summary 프롬프트와 코드는 기존 코드를 사용하여 기능 보존함

따라서 3번째 turn 이후 summary가 생성되는지 확인한다.

In [11]:
from uuid import uuid4

summary_config = thread_config(f"week7-summary-{uuid4().hex}")
summary_questions = [
    "2026년 해외 출장 숙박비 한도와 근거 파일명을 알려줘.",
    "2026년 승인된 출장의 식비 한도와 근거 파일명을 알려줘.",
    "국제 비즈니스 출장 출발 전 승인 요건과 근거 파일명을 알려줘.",
]
summary_results = []
print("[Summary test thread_id]", summary_config["configurable"]["thread_id"])

for index, question in enumerate(summary_questions, start=1):
    print(f"\n{'='*18} Turn {index} {'='*18}")
    # 첫 번째는 새 실행, 이후는 같은 thread의 완료된 State를 이어받는다.
    result = run_approved_turn(
        question, summary_config, continue_thread=(index > 1)
    )
    summary_results.append(result)
    print("Question:", question)
    print("Status:", result["turn_status"])
    print("Saved turns:", len(result["turns"]))
    print("Summary length:", len(result["summary"]))

    assert len(result["turns"]) == index, (
        f"Turn {index} 완료 후 turns={len(result['turns'])}; 예상 값은 {index}입니다."
    )
    assert result["current_task"]["goal"] == question
    assert not app.get_state(summary_config).next, "Graph가 완료되지 않았습니다."



from pprint import pprint

#conversation_state = checkpoint_details(
#    summary_config, "3번째 Turn 완료 후 Checkpoint"
#)
conversation_state  = deepcopy(app.get_state(summary_config).values)
turns = conversation_state["turns"]
summary = conversation_state["summary"]

assert len(turns) == 3
assert summary.strip()

print("\n" + "=" * 60)
print("SUMMARY TEST 결과")
print("=" * 60)

print("\n[1. Turn별 Summary 변화]")
for i, result in enumerate(summary_results, start=1):
    print(f"Turn {i}: Summary 길이 = {len(result['summary'])}")

print("\n[2. 요약 대상인 오래된 대화]")
for i, turn in enumerate(turns[:-2], start=1):
    print(f"\nTurn {i}:")
    pprint(turn, width=100, sort_dicts=False)

print("\n[3. 실제 LLM이 생성한 Summary]")
print("-" * 60)
print(summary)
print("-" * 60)



[Summary test thread_id] week7-summary-cd3c9c24d2534d3b9e2b06b2c25d416e

================== Turn 1 ==================

[계획 검토]
목표: 2026년 해외 출장 숙박비 한도와 근거 파일명을 알려줘.
질문: 이 계획을 승인하거나 수정할 내용을 입력하세요.
1. 검색: 2026 년 해외 출장 숙박비 한도 및 근거 파일명
   depends_on: []
⇒ Command(resume={'approved': True})
Question: 2026년 해외 출장 숙박비 한도와 근거 파일명을 알려줘.
Status: completed
Saved turns: 1
Summary length: 0

================== Turn 2 ==================

[계획 검토]
목표: 2026년 승인된 출장의 식비 한도와 근거 파일명을 알려줘.
질문: 이 계획을 승인하거나 수정할 내용을 입력하세요.
1. search internal company documents for the 2026 approved business trip meal allowance limit and the name of the supporting policy file.
   depends_on: []
⇒ Command(resume={'approved': True})
Question: 2026년 승인된 출장의 식비 한도와 근거 파일명을 알려줘.
Status: completed
Saved turns: 2
Summary length: 0

================== Turn 3 ==================

[계획 검토]
목표: 국제 비즈니스 출장 출발 전 승인 요건과 근거 파일명을 알려줘.
질문: 이 계획을 승인하거나 수정할 내용을 입력하세요.
1. search internal company documents for 'international business trip pre-departur

## 6. 최종 점검


모든 과정 테스트 결과

In [14]:
def event_present(result: dict, name: str) -> bool:
    return any(e.get("event") == name for e in result["current_task"]["trace"])

checks = {
    "도구 범위 밖 질문 실행": weather_result["turn_status"] in ("completed", "failed"),
    "출장 규정 / 계획 승인": event_present(approval_result, "plan_approved"),
    "Checkpoint 대기 상태에 계획 보존": bool(approval_checkpoint_before["current_task"]["plan"]),
    "승인 실행 후 완료 Turn 기록": len(approval_checkpoint_after["turns"]) == 1,
    "자연어 수정 요청": event_present(edit_result, "plan_revision_requested"),
    "LLM 계획 수정": event_present(edit_result, "plan_edited"),
    "수정 계획 재승인": event_present(edit_result, "plan_approved"),
    "수정된 계획 실행": bool(executed_ids) and executed_ids[0] == edit_plan_after[0]["id"],
    "Summary 텍스트 생성": bool(conversation_state["summary"]),
}

print("[기능 검증 결과]")
for label, passed in checks.items():
    print(f"{'PASS' if passed else 'CHECK'} | {label}")
print("\n[최종 응답 상태]")
print("승인:", approval_result["turn_status"])
print("수정:", edit_result["turn_status"])
print("다중 Turn summary:", summary_results[-1]["turn_status"])

[기능 검증 결과]
PASS | 도구 범위 밖 질문 실행
PASS | 출장 규정 / 계획 승인
PASS | Checkpoint 대기 상태에 계획 보존
PASS | 승인 실행 후 완료 Turn 기록
PASS | 자연어 수정 요청
PASS | LLM 계획 수정
PASS | 수정 계획 재승인
PASS | 수정된 계획 실행
PASS | Summary 텍스트 생성

[최종 응답 상태]
승인: completed
수정: completed
다중 Turn summary: completed
